<a href="https://colab.research.google.com/github/adautojr001/AprendizadodeMaquina-Adauto/blob/master/projeto_Agro4_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Esse notebook foi implementado para o trabalho final para a conclusão da disciplina de Supervised Machine Learning do PPGEE do Instituto de Pesquisa Científica da UNESP-Sorocaba. Sob a autoria do aluno espescial Adauto Tadeu de Carvalho Junior e a orientação do professor Alexandre da Silva Simões.

In [ ]:
#Importando as bibliotecas necessárias para a primeira fase do projeto

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Importando o dataset
from google.colab import drive
drive.mount('/content/drive/')
path = '/content/drive/MyDrive/Colab_Notebooks/projeto_Agro4.0/Crop_recommendation.csv'

with open(path, "rt") as f:data= pd.read_csv(path)

In [ ]:
#Visualizando as primeiras 5 linhas
print(data.head())

In [ ]:
#Verificando os valores ausentes
print(data.isnull().sum())

In [ ]:
#Separar variáveis de entrada e saída

X = data.drop('label', axis=1)
y = data['label']

# Dividir o dataset em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#Escalar os dados
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Baseline: KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

print("\n ->KNN Relatório de Classificação:")
print(classification_report(y_test, y_pred_knn))

print(" ->KNN Matriz de Confusão:")
plt.figure(figsize=(8, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_knn), annot=True, fmt='d', cmap='Blues', xticklabels=knn.classes_, yticklabels=knn.classes_)
plt.xlabel('Previsto')
plt.ylabel('Verdadeiro')
plt.title('KNN Matriz de Confusão')
plt.show()

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\n ->Random Forest Relatório de Classificação:")
print(classification_report(y_test, y_pred_rf))

print(" ->Random Forest Matriz de Confusão:")
plt.figure(figsize=(8, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Greens', xticklabels=rf.classes_, yticklabels=rf.classes_)
plt.xlabel('Previsto')
plt.ylabel('Verdadeiro')
plt.title('Random Forest Matriz de Confusão')
plt.show()

# Importância dos Recursos
feature_importances = pd.Series(rf.feature_importances_, index=X.columns)
feature_importances = feature_importances.sort_values(ascending=False)

plt.figure(figsize=(8,4))
sns.barplot(x=feature_importances, y=feature_importances.index,hue=feature_importances.index, palette='viridis')
plt.title('Importância do recurso - Random Forest')
plt.xlabel('Importância')
plt.ylabel('Características')
plt.show()

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
%matplotlib inline
#Treinando Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# Treinando SVM
svm = SVC(kernel='rbf', random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

# Métricas
acc_dt = accuracy_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt, average='weighted')

#Exibindo a validação das métricas Árvore de Decisão
print("\n ->Tree Decision Relatório de Classificação:")
print(classification_report(y_test, y_pred_dt))
print(f"Decision Tree - Acurácia: {acc_dt:.2f}, F1-Score: {f1_dt:.2f}")

acc_svm = accuracy_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')

# Matriz de confusão Decision Tree
plt.figure(figsize=(8,4))
sns.heatmap(confusion_matrix(y_test, y_pred_dt), annot=True, fmt='d', cmap='Blues', xticklabels=dt.classes_, yticklabels=dt.classes_)
plt.title('Decision Tree - Matriz de Confusão')
plt.xlabel('Previsto')
plt.ylabel('Verdadeiro')
plt.show()

#Exibindo a validação das métricas SVM
print("\n ->SVM Relatório de Classificação:")
print(classification_report(y_test, y_pred_svm))
print(f"SVM - Acurácia: {acc_svm:.2f}, F1-Score: {f1_svm:.2f}")

# Matriz de confusão SVM
plt.figure(figsize=(8,4))
sns.heatmap(confusion_matrix(y_test, y_pred_svm), annot=True, fmt='d', cmap='Oranges', xticklabels=dt.classes_, yticklabels=dt.classes_)
plt.title('SVM - Matriz de Confusão')
plt.xlabel('Previsto')
plt.ylabel('Verdadeiro')
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score

#Realizando a validação cruzada utilizando cv=5
#Random Forest
scores_rf = cross_val_score(RandomForestClassifier(random_state=42),
                            X, y, cv=5, scoring='accuracy')
print("Random Forest (Acurácia média com CV=5):", scores_rf.mean())

# Àrvore de Decisão
scores_dt = cross_val_score(DecisionTreeClassifier(random_state=42),
                            X, y, cv=5, scoring='accuracy')
print("Decision Tree (Acurácia média com CV=5):", scores_dt.mean())


# KNN
scores_knn = cross_val_score(KNeighborsClassifier(n_neighbors=5), X, y, cv=5, scoring='accuracy')
print("KNN (Acurácia média com CV=5):", scores_knn.mean())

# SVM
scores_svm = cross_val_score(SVC(kernel='rbf', random_state=42),
                             X, y, cv=5, scoring='accuracy')
print("SVM (Acurácia média com CV=5):", scores_svm.mean())

In [ ]:
#Testando se realmente há diferenças entre as acurácias com ANOVA(one way)
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Random Forest
scores_rf = cross_val_score(RandomForestClassifier(random_state=42), X, y, cv=5, scoring='accuracy')

# Decision Tree
scores_dt = cross_val_score(DecisionTreeClassifier(random_state=42), X, y, cv=5, scoring='accuracy')

# KNN
scores_knn = cross_val_score(KNeighborsClassifier(n_neighbors=5), X, y, cv=5, scoring='accuracy')

# SVM
scores_svm = cross_val_score(SVC(), X, y, cv=5, scoring='accuracy')

In [ ]:
#Rodando o teste
from scipy.stats import f_oneway

# Realiza o teste ANOVA
anova_result = f_oneway(scores_rf, scores_dt, scores_knn, scores_svm)

print("Estatística F:", anova_result.statistic)
print("p-variavel:", anova_result.pvalue)


In [ ]:
# Importando o dataset para a 2 fase do projeto
import pandas as pd

#from google.colab import drive
#drive.mount('/content/drive/')
#path = '/content/drive/MyDrive/Colab_Notebooks/projeto_Agro4.0/PlantVillage'

#with open(path, "rt") as f:data= pd.read_csv(path)

dataset = pd.read_csv("https://www.kaggle.com/datasets/emmarex/plantdisease?select=PlantVillage")
print(dataset.head())

In [ ]:
#Para a 2 fase do projeto iremos carregar as bibliotecas necessárias para criar a nossa DNN...
import tensorflow
from tensorflow.keras.datasets import PlantVillage
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight


dataset = tf.keras.utils.image_dataset_from_directory(
    directory='./PlantVillage',
    labels='inferred',
    label_mode='categorical',
    batch_size=32,
     image_size=(224, 224)
)

with np.load(path) as data:
    X_train = data['x_train']
    Y_train = data['y_train']
    X_test = data['x_test']
    Y_test = data['y_test']

print(X_train.shape)
print(Y_train.shape)
print(X_test.shape)
print(Y_test.shape)